In [2]:
import polars as pl

In [3]:
def show_all(df, width=200, max_col_width=True):
    '''
    Prints an entire polars dataframe in the console or notebook output.
    Parameters
    ----------
    df : pl.DataFrame
        The dataframe to be printed.
    width : int, optional
        The width of the printed dataframe.
        Defaults to 200.
    max_col_width : bool, optional
        Whether to set the maximum column width.
        i.e. it will print the full contents of the cells.
        Defaults to True.
    '''
    with  pl.Config()  as  cfg:
        cfg.set_tbl_cols(-1)
        cfg.set_tbl_rows(-1)
        cfg.set_tbl_width_chars(width)
        if  max_col_width  or  len(df.columns) ==  1:
            cfg.set_fmt_str_lengths(width)
        print(df)

# Gather list of metagenomes we need to process here

In [4]:
# Read in list of all the metagenomes we need to process
# columns = [.acc,.releasedate,.mbases,.organism,.mbytes]
columns = ['acc','releasedate','organism','mbytes','avespotlen','bases','librarylayout']
all_metags = pl.read_csv('~/git/sandpiper/sra_metadata/sra_metadata_20260303.some_columns.csv.gz', has_header=False)
all_metags.columns = columns
all_metags.head(), all_metags.shape

(shape: (5, 7)
 ┌─────────────┬────────────────┬────────────────┬────────┬────────────┬────────────┬───────────────┐
 │ acc         ┆ releasedate    ┆ organism       ┆ mbytes ┆ avespotlen ┆ bases      ┆ librarylayout │
 │ ---         ┆ ---            ┆ ---            ┆ ---    ┆ ---        ┆ ---        ┆ ---           │
 │ str         ┆ str            ┆ str            ┆ i64    ┆ i64        ┆ i64        ┆ str           │
 ╞═════════════╪════════════════╪════════════════╪════════╪════════════╪════════════╪═══════════════╡
 │ ERR15880073 ┆ 2025-11-19T00: ┆ wastewater     ┆ 810    ┆ 216        ┆ 2651825214 ┆ PAIRED        │
 │             ┆ 00:00+00:00    ┆ metagenome     ┆        ┆            ┆            ┆               │
 │ SRR16203935 ┆ 2021-10-05T00: ┆ tick           ┆ 769    ┆ 302        ┆ 1825027978 ┆ PAIRED        │
 │             ┆ 00:00+00:00    ┆ metagenome     ┆        ┆            ┆            ┆               │
 │ SRR34723171 ┆ 2025-12-01T00: ┆ human          ┆ 2440   ┆ 302    

In [5]:
# Read in list of metagenomes already processed
prev = pl.read_csv('~/m/msingle/mess/193_202502_sra_scrape/find_gz20250402.accessions.uniq.txt', has_header=False)
prev.columns = ['acc']
prev.head(), prev.shape

(shape: (5, 1)
 ┌───────────┐
 │ acc       │
 │ ---       │
 │ str       │
 ╞═══════════╡
 │ DRR000713 │
 │ DRR000714 │
 │ DRR001356 │
 │ DRR001360 │
 │ DRR001455 │
 └───────────┘,
 (781079, 1))

In [6]:
# Subtract the two to get the list of metagenomes we still need to process
to_process = all_metags.filter(~pl.col('acc').is_in(prev['acc']))
to_process.head(), to_process.shape

(shape: (5, 7)
 ┌─────────────┬────────────────┬───────────────┬────────┬────────────┬─────────────┬───────────────┐
 │ acc         ┆ releasedate    ┆ organism      ┆ mbytes ┆ avespotlen ┆ bases       ┆ librarylayout │
 │ ---         ┆ ---            ┆ ---           ┆ ---    ┆ ---        ┆ ---         ┆ ---           │
 │ str         ┆ str            ┆ str           ┆ i64    ┆ i64        ┆ i64         ┆ str           │
 ╞═════════════╪════════════════╪═══════════════╪════════╪════════════╪═════════════╪═══════════════╡
 │ ERR15880073 ┆ 2025-11-19T00: ┆ wastewater    ┆ 810    ┆ 216        ┆ 2651825214  ┆ PAIRED        │
 │             ┆ 00:00+00:00    ┆ metagenome    ┆        ┆            ┆             ┆               │
 │ SRR34723171 ┆ 2025-12-01T00: ┆ human         ┆ 2440   ┆ 302        ┆ 7295550202  ┆ PAIRED        │
 │             ┆ 00:00+00:00    ┆ respiratory   ┆        ┆            ┆             ┆               │
 │             ┆                ┆ syncytial vi… ┆        ┆         

In [7]:
# How many are from before 2025? ~4% or something
to_process.group_by(pl.col('releasedate') < '2025-02-01').len()

releasedate,len
bool,u32
false,293596
true,12183


In [8]:
293596+707470 # 984575 #=> just over 1 million, but it won't make it.

1001066

# batch0 testing chunking

In [29]:
import os
os.makedirs('runlists_20260219', exist_ok=True)

def generate_and_write_batch(
    batch_name,
    to_process,
    blacklist_accs,
    making_batch,
    num_acc_to_select,
    gbp_per_chunk=5,
):
    print("Overal status ====")
    print(f"Blacklist (finished?) was {len(blacklist_accs)} accs, {len(to_process) - len(blacklist_accs)} {(1-len(blacklist_accs)/len(to_process))*100:.2f}% left to process")

    batch_possible = to_process.filter(~pl.col('acc').is_in(blacklist_accs))

    batch_possible = batch_possible.with_columns((gbp_per_chunk / pl.col('bases_per_read') * 1e9).ceil().cast(pl.Int32).alias('chunk_size1'))
    # Ensure batch size is even, otherwise singlem complains. Add 1 to odd chunk sizes.
    batch_possible = batch_possible.with_columns((pl.col('chunk_size1') + (pl.col('chunk_size1') % 2)).alias('chunk_size'))
    batch_possible = batch_possible.with_columns((pl.col('bases') / (pl.col('chunk_size') * pl.col('bases_per_read'))).ceil().cast(pl.Int32).alias('num_chunks').cast(pl.Int32))
    batch_possible = batch_possible.filter((pl.col('bases') < 30e9))

    print("possibles ====")
    print(batch_possible.shape)

    if making_batch:
        batch = batch_possible.sample(num_acc_to_select, seed=42)
        print("chosen acc-wise ====")
        print(batch.head(), batch.shape)

        batch_exploded = batch.group_by('acc','num_chunks','chunk_size').agg(pl.arange(1, pl.col('num_chunks')+1).alias('chunk')).explode('chunk')
        # randomise order
        batch_exploded = batch_exploded.sample(fraction=1, seed=42, shuffle=True)
        print("exploded ====")
        batch_exploded = batch_exploded.select(['acc',pl.lit(batch_name).alias('batch_name'),'chunk','chunk_size'])
        print(batch_exploded[:10], batch_exploded.shape)

        batch_exploded.write_csv(f'runlists_20260219/{batch_name}.csv')

    # Read it back in and return
    batch_exploded = pl.read_csv(f'runlists_20260219/{batch_name}.csv')
    return batch_exploded

batch0_exploded = generate_and_write_batch(
    batch_name='batch0',
    to_process=to_process.filter(pl.col('acc')=='SRR26138773'),
    blacklist_accs=[],
    making_batch=True,
    num_acc_to_select=1,
)

Overal status ====
Blacklist (finished?) was 0 accs, 1 100.00% left to process
possibles ====
(1, 11)
chosen acc-wise ====
shape: (1, 11)
┌────────────┬────────────┬───────────┬────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ acc        ┆ releasedat ┆ organism  ┆ mbytes ┆ … ┆ bases_per ┆ chunk_siz ┆ chunk_siz ┆ num_chunk │
│ ---        ┆ e          ┆ ---       ┆ ---    ┆   ┆ _read     ┆ e1        ┆ e         ┆ s         │
│ str        ┆ ---        ┆ str       ┆ i64    ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│            ┆ str        ┆           ┆        ┆   ┆ f64       ┆ i32       ┆ i32       ┆ i32       │
╞════════════╪════════════╪═══════════╪════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ SRR2613877 ┆ 2025-07-01 ┆ feces met ┆ 3527   ┆ … ┆ 149.5     ┆ 33444817  ┆ 33444818  ┆ 3         │
│ 3          ┆ T00:00:00+ ┆ agenome   ┆        ┆   ┆           ┆           ┆           ┆           │
│            ┆ 00:00      ┆           ┆        ┆   ┆  

# batch1

In [30]:
batch1_exploded = generate_and_write_batch(
    batch_name='batch1',
    to_process=to_process.filter((pl.col('bases') < 30e9)),
    blacklist_accs=[],
    num_acc_to_select=100,
    making_batch=True,
)

# # First choose 100 random ones, and chunk them into 5Gbp chunks. But only those <30Gbp so nodes don't get too filled up disk space wise on download.
# gbp_per_chunk = 5
# # When the librarylayout is SINGLE, we are OK. When it's PAIRED, we need to half the chunk size because 1 spot will contain 2 reads.
# to_process = to_process.with_columns(pl.when(pl.col('librarylayout') == 'PAIRED').then(pl.col('avespotlen') / 2).otherwise(pl.col('avespotlen')).alias('bases_per_read'))

# batch1_possible = to_process.with_columns((gbp_per_chunk / pl.col('bases_per_read') * 1e9).ceil().cast(pl.Int32).alias('chunk_size1'))
# # Ensure batch size is even, otherwise singlem complains. Add 1 to odd chunk sizes.
# batch1_possible = batch1_possible.with_columns((pl.col('chunk_size1') + (pl.col('chunk_size1') % 2)).alias('chunk_size'))
# batch1_possible = batch1_possible.with_columns((pl.col('bases') / (pl.col('chunk_size') * pl.col('bases_per_read'))).ceil().cast(pl.Int32).alias('num_chunks').cast(pl.Int32))
# batch1_possible = batch1_possible.filter((pl.col('bases') < 30e9))
# batch1_possible.shape, batch1_possible[:4]

Overal status ====
Blacklist (finished?) was 0 accs, 296517 100.00% left to process
possibles ====
(296517, 11)
chosen acc-wise ====
shape: (5, 11)
┌────────────┬────────────┬───────────┬────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ acc        ┆ releasedat ┆ organism  ┆ mbytes ┆ … ┆ bases_per ┆ chunk_siz ┆ chunk_siz ┆ num_chunk │
│ ---        ┆ e          ┆ ---       ┆ ---    ┆   ┆ _read     ┆ e1        ┆ e         ┆ s         │
│ str        ┆ ---        ┆ str       ┆ i64    ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│            ┆ str        ┆           ┆        ┆   ┆ f64       ┆ i32       ┆ i32       ┆ i32       │
╞════════════╪════════════╪═══════════╪════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ SRR3260117 ┆ 2025-03-06 ┆ freshwate ┆ 2664   ┆ … ┆ 149.0     ┆ 33557047  ┆ 33557048  ┆ 2         │
│ 5          ┆ T00:00:00+ ┆ r metagen ┆        ┆   ┆           ┆           ┆           ┆           │
│            ┆ 00:00      ┆ ome       ┆     

In [ ]:
# batch1_exploded.filter(pl.col('acc')=='ERR14334714'), batch1_exploded.filter(pl.col('acc')=='ERR14334714')['chunk_size'][0]
# 
# # 8333334 reads per chunk. Each read is 150np. So with 8 chunks we get 8333334*150*8
33333334*150*4/1e9 #=> 20G, which is right. We need 18G total.

20.0000004

# batch2 - another 1000 jobs in batch2 (but also randomise their order so not the same input first)

In [32]:
previous_accs_here = pl.concat([
    batch1_exploded
]).select(pl.col('acc').unique())['acc'].to_list()

batch2_exploded = generate_and_write_batch(
    batch_name='batch2',
    to_process=to_process.filter(pl.col('bases') < 30e9),
    blacklist_accs=previous_accs_here,
    making_batch=True,
    num_acc_to_select=1000,
)

# batch2_possible = to_process.filter(~pl.col('acc').is_in(batch1_accs))

# batch2_possible = batch2_possible.with_columns((gbp_per_chunk / pl.col('bases_per_read') * 1e9).ceil().cast(pl.Int32).alias('chunk_size1'))
# # Ensure batch size is even, otherwise singlem complains. Add 1 to odd chunk sizes.
# batch2_possible = batch2_possible.with_columns((pl.col('chunk_size1') + (pl.col('chunk_size1') % 2)).alias('chunk_size'))
# batch2_possible = batch2_possible.with_columns((pl.col('bases') / (pl.col('chunk_size') * pl.col('bases_per_read'))).ceil().cast(pl.Int32).alias('num_chunks').cast(pl.Int32))
# batch2_possible = batch2_possible.filter((pl.col('bases') < 30e9))

# print("possibles ====")
# print(batch2_possible.head(), batch2_possible.shape)

# making_batch2 = True
# if making_batch2:
#     batch2 = batch2_possible.sample(1000, seed=42)
#     print("chosen acc-wise ====")
#     print(batch2.head(), batch2.shape)

#     batch2_exploded = batch2.group_by('acc','avespotlen','num_chunks','chunk_size').agg(pl.arange(1, pl.col('num_chunks')+1).alias('chunk')).explode('chunk')
#     # randomise order
#     batch2_exploded = batch2_exploded.sample(fraction=1, seed=42, shuffle=True)
#     print("exploded ====")
#     print(batch2_exploded[:10], batch2_exploded.shape)

#     batch2_exploded.write_csv('runlists_20260219/batch2.csv')

Overal status ====
Blacklist (finished?) was 100 accs, 296417 99.97% left to process
possibles ====
(296417, 11)
chosen acc-wise ====
shape: (5, 11)
┌────────────┬────────────┬───────────┬────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ acc        ┆ releasedat ┆ organism  ┆ mbytes ┆ … ┆ bases_per ┆ chunk_siz ┆ chunk_siz ┆ num_chunk │
│ ---        ┆ e          ┆ ---       ┆ ---    ┆   ┆ _read     ┆ e1        ┆ e         ┆ s         │
│ str        ┆ ---        ┆ str       ┆ i64    ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│            ┆ str        ┆           ┆        ┆   ┆ f64       ┆ i32       ┆ i32       ┆ i32       │
╞════════════╪════════════╪═══════════╪════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ SRR3247949 ┆ 2025-12-18 ┆ human met ┆ 308    ┆ … ┆ 138.0     ┆ 36231885  ┆ 36231886  ┆ 1         │
│ 8          ┆ T00:00:00+ ┆ agenome   ┆        ┆   ┆           ┆           ┆           ┆           │
│            ┆ 00:00      ┆           ┆    